In [1]:
"""
@Modified by: Zilan Cheng"""

import torch.nn.functional as F
from timeit import default_timer
from utilities3 import *
import numpy as np
import matplotlib.pyplot as plt

torch.cuda.set_device(0)
torch.manual_seed(0)
np.random.seed(0)

In [2]:
class SpectralConv3d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super(SpectralConv3d, self).__init__()

        """
        3D Fourier layer. It does FFT, linear transform, and Inverse FFT.    
        """

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1 
        self.modes2 = modes2

        self.scale = (1 / (in_channels * out_channels))
        self.weights1 = nn.Parameter(self.scale * torch.rand(self.in_channels, self.out_channels, self.modes1, self.modes2,2, dtype=torch.cfloat))
        self.weights2 = nn.Parameter(self.scale * torch.rand(self.in_channels, self.out_channels, self.modes1, self.modes2,2, dtype=torch.cfloat))

    def compl_mul2d(self, input, weights):
        return torch.einsum("bixyz,ioxyz->boxyz", input, weights)

    def forward(self, x):
        batchsize = x.shape[0]
        x_ft = torch.fft.rfft2(x,dim=(-3,-2))

        out_ft = torch.zeros(batchsize, self.out_channels,  x.size(-3), x.size(-2)//2 + 1, 2, dtype=torch.cfloat, device=x.device)
        out_ft[:, :, :self.modes1, :self.modes2,:] = \
            self.compl_mul2d(x_ft[:, :, :self.modes1, :self.modes2,:], self.weights1)
        out_ft[:, :, -self.modes1:, :self.modes2,:] = \
            self.compl_mul2d(x_ft[:, :, -self.modes1:, :self.modes2,:], self.weights2)

        x = torch.fft.irfft2(out_ft, s=(x.size(-3), x.size(-2)),dim=(-3,-2))
        return x

In [3]:
class MLP(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels):
        super(MLP, self).__init__()
        self.mlp1 = nn.Conv3d(in_channels, mid_channels, 1)
        self.mlp2 = nn.Conv3d(mid_channels, out_channels, 1)

    def forward(self, x):
        x = self.mlp1(x)
        x = F.gelu(x)
        x = self.mlp2(x)
        return x

In [4]:
#Constant variables injected to biases
class MLP_t_theta(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels,t_channels,theta_channels):
        super(MLP_t_theta, self).__init__()
        self.mlp = nn.Conv3d(in_channels, out_channels, 1)
        self.theta_emb1 = nn.Linear(theta_channels, out_channels)
        self.theta_emb2 = nn.Linear(out_channels, out_channels)
        self.theta_act = F.gelu

    def forward(self, x, theta):
        x = self.mlp(x) 
        x = x + self.theta_emb2(self.theta_act(self.theta_emb1(theta)))[:, :, None, None, None] #
        return x

In [5]:
def get_grid(shape, device):
    batchsize, size_x, size_y = shape[0], shape[1], shape[2]
    gridx = torch.tensor(np.linspace(0, 1, size_x), dtype=torch.float)
    gridx = gridx.reshape(1, size_x, 1, 1, 1).repeat([batchsize, 1, size_y, 2, 1])
    gridy = torch.tensor(np.linspace(0, 1, size_y), dtype=torch.float)
    gridy = gridy.reshape(1, 1, size_y, 1, 1).repeat([batchsize, size_x, 1, 2, 1])
    return torch.cat((gridx, gridy), dim=-1).to(device)

In [6]:
class FNO3d(nn.Module):
    def __init__(self, modes,width,theta_channels = 1):
        super(FNO3d, self).__init__()

        self.modes = modes
        self.width = width
        x_channels = 3
        t_channels =1
        
        self.p = nn.Linear(3, self.width)
        self.conv0 = SpectralConv3d(self.width,self.width, self.modes,self.modes )
        self.conv1 = SpectralConv3d(self.width,self.width, self.modes,self.modes )
        self.conv2 = SpectralConv3d(self.width,self.width, self.modes,self.modes )
        self.conv3 = SpectralConv3d(self.width,self.width, self.modes,self.modes )
        self.mlp0 = MLP_t_theta(self.width, self.width, self.width,t_channels,theta_channels)
        self.mlp1 = MLP_t_theta(self.width, self.width, self.width,t_channels,theta_channels)
        self.mlp2 = MLP_t_theta(self.width, self.width, self.width,t_channels,theta_channels)
        self.mlp3 = MLP_t_theta(self.width, self.width, self.width,t_channels,theta_channels)
        self.w0 = nn.Conv3d(self.width, self.width, 1)
        self.w1 = nn.Conv3d(self.width, self.width, 1)
        self.w2 = nn.Conv3d(self.width, self.width, 1)
        self.w3 = nn.Conv3d(self.width, self.width, 1)
        self.q = MLP(self.width, 1, self.width * 4)

    def forward(self, x,theta):
        grid = get_grid(x.shape, x.device)
        x = torch.cat((x, grid), dim=-1)
        x=x.to(torch.float32)
        x = self.p(x)
        x = x.permute(0, 4, 1, 2, 3)
        x1 = self.conv0(x)
        x1 = self.mlp0(x1,theta)
        x2 = self.w0(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv1(x)
        x1 = self.mlp1(x1,theta)
        x2 = self.w1(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv2(x)
        x1 = self.mlp2(x1,theta)
        x2 = self.w2(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv3(x)
        x1 = self.mlp3(x1,theta)
        x2 = self.w3(x)
        x = x1 + x2

        x = self.q(x)
        x = x.permute(0, 2, 3, 4, 1)
        return x

In [7]:
################################################################
# configs
################################################################

ntrain = 900
ntest = 100
nsnap =900

width = 32
modes =6

mesh=64
r=1

batch_size = 20
learning_rate = 0.001
epochs = 500
iterations = epochs*(ntrain//batch_size)

In [8]:
################################################################
# dataloader
################################################################
dataloader=MatReader('../data/nls/pNLS_data.mat')

u_re1=dataloader.read_field('u_re1')
u_im1=dataloader.read_field('u_im1')
u_re2=dataloader.read_field('u_re2')
u_im2=dataloader.read_field('u_im2')
theta_train=dataloader.read_field('p1')[:,:ntrain].permute(1,0)
theta_test=dataloader.read_field('p1')[:,-ntest:].permute(1,0)

u0_train_re=u_re1[:,:,:ntrain][::r,::r,:]
u0_train_im=u_im1[:,:,:ntrain][::r,::r,:]
u_end_train_re=u_re2[:,:,:ntrain][::r,::r,:]
u_end_train_im=u_im2[:,:,:ntrain][::r,::r,:]
u0_test_re=u_re1[:,:,-ntest:][::r,::r,:]
u0_test_im=u_im1[:,:,-ntest:][::r,::r,:]
u_end_test_re=u_re2[:,:,-ntest:][::r,::r,:]
u_end_test_im=u_im2[:,:,-ntest:][::r,::r,:]

u_train_0=torch.stack((u0_train_re,u0_train_im),dim=0)
u_test_0=torch.stack((u0_test_re,u0_test_im),dim=0)
u_train_end=torch.stack((u_end_train_re,u_end_train_im),dim=0)
u_test_end=torch.stack((u_end_test_re,u_end_test_im),dim=0)

x_train=u_train_0.permute(3,1,2,0)
x_test=u_test_0.permute(3,1,2,0)
y_train=u_train_end.permute(3,1,2,0)
y_test=u_test_end.permute(3,1,2,0)

x_train = x_train.reshape(ntrain,mesh,mesh,2,1)
x_test = x_test.reshape(ntest,mesh,mesh,2,1)
y_train = y_train.reshape(ntrain,mesh,mesh,2,1)
y_test = y_test.reshape(ntest,mesh,mesh,2,1)

x_normalizer = UnitGaussianNormalizer(x_train)
x_train = x_normalizer.encode(x_train)
x_test = x_normalizer.encode(x_test)

theta_normalizer = UnitGaussianNormalizer(theta_train)
theta_train = theta_normalizer.encode(theta_train)
theta_test = theta_normalizer.encode(theta_test)

y_normalizer = UnitGaussianNormalizer(y_train)
y_train = y_normalizer.encode(y_train)

train_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_train, theta_train,y_train), batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_test, theta_test, y_test), batch_size=batch_size, shuffle=False)

d1=mesh
d2=mesh
d3=2

In [9]:
solution_real=torch.zeros(ntest,mesh,mesh,2,1)
solution_learnt=torch.zeros(ntest,mesh,mesh,2,1)

In [10]:
################################################################
# training and evaluation
################################################################
model = FNO3d(modes,width,theta_channels = 1).to(device)
print(count_params(model))

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=iterations)

myloss = LpLoss(size_average=False)
y_normalizer.to(device)
for ep in range(epochs):
    i=0
    model.train()
    t1 = default_timer()
    train_l2 = 0
    for x, theta,  y in train_loader:
        x, theta,  y = x.to(device), theta.to(device),y.to(device)

        optimizer.zero_grad()
        out = model(x,theta).reshape(batch_size, d1, d2, d3,1)
        out = y_normalizer.decode(out)
        y = y_normalizer.decode(y)

        loss =myloss(out.reshape(batch_size,d1*d2*d3),y.reshape(batch_size,d1*d2*d3))
        loss.backward()

        optimizer.step()
        scheduler.step()
        train_l2 += loss.item()

    model.eval()
    t2 =default_timer()
    test_l2 = 0.0
    with torch.no_grad():
        for x, theta, y in test_loader:
            x, theta, y = x.to(device), theta.to(device),y.to(device)

            out = model(x,theta).reshape(batch_size, d1,d2,d3,1)
            out = y_normalizer.decode(out)
            solution_real[i:i+batch_size,:,:,:,:]=y
            solution_learnt[i:i+batch_size,:,:,:,:]=out
            i=i+batch_size
            test_l2 += myloss(out.view(batch_size,d1*d2*d3), y.view(batch_size,d1*d2*d3)).item()
    t3 = default_timer()
    train_l2/= ntrain
    test_l2 /= ntest

    print(ep, t2-t1,t3-t2,train_l2, test_l2)


1197057
0 3.869432996958494 0.20958111435174942 0.6899479590521919 0.5763010692596435
1 2.7369712749496102 0.1339162504300475 0.4798879199557834 0.34728667736053465
2 3.1938799330964684 0.4266087803989649 0.3042833052741157 0.2621063375473022
3 3.366634087637067 0.23579790070652962 0.24654599640104505 0.2240874719619751
4 3.737719714641571 0.14503971859812737 0.21261012660132514 0.19556617259979248
5 3.1701894486323 0.1342721488326788 0.18971095403035482 0.17761385679244995
6 3.6945902202278376 0.1644160207360983 0.17595990631315445 0.16807825803756715
7 3.698536043986678 0.3979861829429865 0.16391755633884006 0.15496822357177734
8 3.646452309563756 0.12368126958608627 0.15598287502924602 0.1480679702758789
9 2.9909177711233497 0.3360787881538272 0.15024121787812975 0.14393040657043457
10 3.852080120705068 0.17332148924469948 0.1463226185904609 0.14228322505950927
11 3.851172990165651 0.22212395817041397 0.14062473667992487 0.13588563680648805
12 3.3468493362888694 0.3852516319602728 0